In [2]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_community.document_loaders import UnstructuredFileLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.storage import LocalFileStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda


llm = ChatOpenAI(
    model="gpt-5-nano",
    temperature=0.1
)

cache_dir = LocalFileStore("./.cache/")



splitter = CharacterTextSplitter.from_tiktoken_encoder(
    model_name="gpt-5-nano",
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)

loader = UnstructuredFileLoader("./files/chapter_one.docx")

docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()

cache_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embeddings, cache_dir
)

vectorstore = FAISS.from_documents(docs, cache_embeddings)

retriever = vectorstore.as_retriever() 

map_doc_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            Use the following portion of a long document to see if any of the
            text is relevant to answer the question. return any relevant text
            verbatim.
            -------
            {context}
            """
        ),
        ("human", "{question}")
    ]
)

map_doc_chain = map_doc_prompt | llm

def map_docs(inputs):
    documents = inputs['documents']
    question = inputs['question']
    return "\n\n".join(map_doc_chain.invoke({
        "context":doc.page_content,
        "question": question
    }).content for doc in documents)

map_chain = {"documents": retriever, "question": RunnablePassthrough()} | RunnableLambda(map_docs)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", """You are a helpful assistant. Asnwer questions using only the following context.
        If you don't know the answer just say you don't know,
        don't make it up:\n\n{context}"""),
        ("human", "{question}")
    ]
)

chain = {"context": map_chain ,"question": RunnablePassthrough()} | final_prompt | llm

chain.invoke("Where does Winston go to work")

AIMessage(content='He works at the Ministry of Truth, in the Records Department.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 470, 'prompt_tokens': 246, 'total_tokens': 716, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dc7MXmsiy44qjFfa0oDPiOjrpAxjq', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019df7b2-1259-78e0-8ac8-a6b1a4ede26d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 246, 'output_tokens': 470, 'total_tokens': 716, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 448}})